# Check joined tables

Verifies that buildJoinedObjectTable correctly merges object statistics with per-catalog source columns.

In [ ]:
# Standard imports
import tables_io
import hpmcm
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
tract = 8006

# Input catalogs and their IDs for the meta-match
all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_stats.pq',
    f'{tract}/rubin_md_match_object_stats.pq',
]
all_ids = list(range(len(all_files)))

In [ ]:
# Load association and stats tables
object_assoc = hpmcm.output_tables.ObjectAssocTable.read(f'{tract}/all_match_object_assoc.pq', [])
object_stats = hpmcm.output_tables.ObjectStatsTable.read(f'{tract}/all_match_object_stats.pq', [])

In [ ]:
# Build wide joined table — one row per object, source cols suffixed by catalog_id
joined = hpmcm.output_tables.buildJoinedObjectTable(
    object_assoc,
    all_files,
    all_ids,
)

In [ ]:
# Count objects detected in catalog 0 (bit 0 set)
(np.bitwise_and(joined['catalog_mask'],1) == 1).sum()

In [ ]:
len(joined['catalog_mask'])

In [ ]:
len(object_assoc)

In [ ]:
len(object_stats)

In [ ]:
joined

In [ ]:
object_stats

In [ ]:
object_stats.index = object_stats['object_id']

In [ ]:
oo = object_stats.join(joined, lsuffix='_obj', rsuffix='_assoc')

In [ ]:
oo['snr_1'][np.isfinite(oo['snr_1'])]

In [ ]:
# Histogram of fluxes split by which catalogs detected each object
for i in range(1,16):
    _ = plt.hist(joined['g_gaapPsfFlux_0'][joined['catalog_mask']==i], bins=np.logspace(0, 6, 61))
_ = plt.xscale('log')
_ = plt.yscale('log')